# Dataset versioning with lakeFS on Backblaze B2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/backblaze-b2-samples/notebooks/blob/main/lakefs-b2-dataset-versioning/lakefs_b2_dataset_versioning.ipynb) [![Open In Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/backblaze-b2-samples/notebooks/HEAD?urlpath=lab/tree/lakefs-b2-dataset-versioning/lakefs_b2_dataset_versioning.ipynb) [![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/backblaze-b2-samples/notebooks?quickstart=1)

This notebook walks through a satellite-imagery dataset versioning scenario, using lakeFS as the data versioning layer and Backblaze B2 as the underlying object storage:

1. Generate a synthetic catalog of satellite tiles with per-tile band statistics and a `cloud_mask_v1` label.
2. Commit the catalog to `main` in a fresh lakeFS repo.
3. Branch into `experiment-cloud-mask-v2`, rerun a stricter classifier, commit the modified catalog.
4. Diff the branch against `main` to inspect what changed.
5. Merge the experiment back into `main`, then revert the merge to show that lakeFS keeps both the merge and the revert in history.
6. (Optional) Peek at the B2 bucket directly to see the content-addressed layout lakeFS produces.

**Prerequisite.** A running lakeFS server pointed at a B2 bucket. The simplest path is the companion sample app at <https://github.com/backblaze-b2-samples/lakefs-on-b2-quickstart>; clone it, `cp .env.example .env`, fill in your B2 details, `docker compose up -d`, and lakeFS will be listening on `http://localhost:8000`.

## Setup

In [ ]:
%pip install -q boto3>=1.34 lakefs>=0.7 numpy>=1.26 pandas>=2.1 pyarrow>=15.0

In [ ]:
import getpass
import io
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import lakefs
from lakefs.client import Client
from lakefs.exceptions import NotFoundException

RNG = np.random.default_rng(seed=20260528)
REPO_NAME = os.environ.get("LAKEFS_NB_REPO_NAME", "").strip() or "satellite-tiles"
EXPERIMENT_BRANCH = "experiment-cloud-mask-v2"
NUM_TILES = int(os.environ.get("LAKEFS_NB_NUM_TILES", "1000"))
print(f"Will generate {NUM_TILES} synthetic tiles into repo {REPO_NAME!r}.")

### lakeFS connection

The notebook reads `LAKEFS_ENDPOINT_URL`, `LAKEFS_ACCESS_KEY_ID`, and `LAKEFS_SECRET_ACCESS_KEY` from the environment first. If any is missing it falls back to an interactive prompt. Use the values from the `lakefs-on-b2-quickstart` sample's `.env` file.

In [ ]:
LAKEFS_ENDPOINT_URL = os.environ.get("LAKEFS_ENDPOINT_URL", "").strip()
if not LAKEFS_ENDPOINT_URL:
    LAKEFS_ENDPOINT_URL = input("lakeFS endpoint URL [http://localhost:8000]: ").strip() or "http://localhost:8000"

LAKEFS_ACCESS_KEY_ID = os.environ.get("LAKEFS_ACCESS_KEY_ID", "").strip()
if not LAKEFS_ACCESS_KEY_ID:
    LAKEFS_ACCESS_KEY_ID = input("lakeFS access key ID: ").strip()

LAKEFS_SECRET_ACCESS_KEY = os.environ.get("LAKEFS_SECRET_ACCESS_KEY", "").strip()
if not LAKEFS_SECRET_ACCESS_KEY:
    LAKEFS_SECRET_ACCESS_KEY = getpass.getpass("lakeFS secret access key: ").strip()

lakefs_client = Client(
    host=LAKEFS_ENDPOINT_URL,
    username=LAKEFS_ACCESS_KEY_ID,
    password=LAKEFS_SECRET_ACCESS_KEY,
)
print(f"Connected to lakeFS: {lakefs_client.version}")

### B2 bucket configuration

Needed by the optional verification cell at the end of the notebook. If you skip the verification cell you can leave these unset.

In [ ]:
LAKEFS_NB_STORAGE_NAMESPACE = os.environ.get("LAKEFS_NB_STORAGE_NAMESPACE", "").strip()
B2_LAKEFS_DATA_BUCKET = os.environ.get("B2_LAKEFS_DATA_BUCKET", "").strip()
if not B2_LAKEFS_DATA_BUCKET and not LAKEFS_NB_STORAGE_NAMESPACE:
    B2_LAKEFS_DATA_BUCKET = input("B2 bucket name for the lakeFS storage namespace: ").strip()

PRIVATE_B2_REGION = os.environ.get("PRIVATE_B2_REGION", "").strip()
AWS_ENDPOINT_URL_S3 = os.environ.get("AWS_ENDPOINT_URL_S3", "").strip()
if B2_LAKEFS_DATA_BUCKET and not PRIVATE_B2_REGION:
    # No silent default: B2 returns a misleading 'InvalidAccessKeyId' on region
    # mismatch (see CLAUDE.md region gotcha). Ask for the region explicitly.
    PRIVATE_B2_REGION = input(
        "B2 bucket region (e.g. us-east-005, us-west-001, us-west-004): "
    ).strip()
if not AWS_ENDPOINT_URL_S3 and PRIVATE_B2_REGION:
    AWS_ENDPOINT_URL_S3 = f"https://s3.{PRIVATE_B2_REGION}.backblazeb2.com"
STORAGE_NAMESPACE = (
    LAKEFS_NB_STORAGE_NAMESPACE
    or (f"s3://{B2_LAKEFS_DATA_BUCKET}/{REPO_NAME}" if B2_LAKEFS_DATA_BUCKET else "")
)
print(f"B2 bucket: {B2_LAKEFS_DATA_BUCKET or '(skipped)'}")
print(f"B2 endpoint: {AWS_ENDPOINT_URL_S3 or '(skipped)'}")
print(f"B2 region: {PRIVATE_B2_REGION or '(skipped)'}")
print(f"Storage namespace: {STORAGE_NAMESPACE or '(skipped)'}")

## 1. Generate a synthetic satellite tile catalog

Each row represents one satellite image tile with per-band reflectance statistics and a `cloud_mask_v1` label produced by a hypothetical first-generation cloud classifier. Real-world tile catalogs include geometries, sensor metadata, and processing-history columns; the shape here is intentionally minimal to keep the demo focused on lakeFS branching mechanics.

In [ ]:
def make_catalog(n_tiles: int, rng: np.random.Generator) -> pd.DataFrame:
    """Return a DataFrame with one row per synthetic satellite tile."""
    tile_ids = [f"T{idx:06d}" for idx in range(n_tiles)]
    bands = rng.uniform(low=0.02, high=0.85, size=(n_tiles, 4))
    cloud_mask_v1 = (bands[:, 0] > 0.6).astype("int8")
    return pd.DataFrame({
        "tile_id": tile_ids,
        "lat": rng.uniform(low=-60.0, high=60.0, size=n_tiles).astype("float32"),
        "lon": rng.uniform(low=-180.0, high=180.0, size=n_tiles).astype("float32"),
        "band_blue": bands[:, 0].astype("float32"),
        "band_green": bands[:, 1].astype("float32"),
        "band_red": bands[:, 2].astype("float32"),
        "band_nir": bands[:, 3].astype("float32"),
        "cloud_mask_v1": cloud_mask_v1,
    })


def df_to_parquet_bytes(df: pd.DataFrame) -> bytes:
    """Serialize a DataFrame to Parquet, returning the bytes."""
    buf = io.BytesIO()
    pq.write_table(pa.Table.from_pandas(df), buf, compression="snappy")
    return buf.getvalue()


catalog_v1 = make_catalog(NUM_TILES, RNG)
print(f"catalog_v1: {len(catalog_v1)} tiles, {catalog_v1['cloud_mask_v1'].sum()} flagged cloudy")
catalog_v1.head()

## 2. Create a repo and commit the catalog to `main`

The storage namespace tells lakeFS where to lay out the actual bytes in B2. By default this notebook uses `s3://${B2_LAKEFS_DATA_BUCKET}/${REPO_NAME}`. CI overrides it via `LAKEFS_NB_STORAGE_NAMESPACE` so each run lands under an isolated prefix.

In [ ]:
if not STORAGE_NAMESPACE:
    raise RuntimeError(
        "STORAGE_NAMESPACE is empty. "
        "Set B2_LAKEFS_DATA_BUCKET (and optionally LAKEFS_NB_STORAGE_NAMESPACE)."
    )
# exist_ok=True returns the existing repo instead of raising if it already exists.
repo = lakefs.repository(REPO_NAME, client=lakefs_client).create(
    storage_namespace=STORAGE_NAMESPACE,
    default_branch="main",
    exist_ok=True,
)
print(f"Repository {REPO_NAME} ready (storage: {STORAGE_NAMESPACE})")

main = repo.branch("main")
main.object("tiles/catalog.parquet").upload(data=df_to_parquet_bytes(catalog_v1))
initial_commit = main.commit(message="Initial catalog with cloud_mask_v1")
print(f"main now at commit {initial_commit.get_commit().id[:10]}")

## 3. Branch and run a stricter cloud-mask classifier

Real-world flow: a data scientist wants to test a new classifier without disturbing the production catalog. They branch from `main`, regenerate the catalog with `cloud_mask_v2`, and commit. The `main` branch is untouched the entire time.

In [ ]:
experiment = repo.branch(EXPERIMENT_BRANCH)
try:
    experiment.delete()
    print(f"Deleted existing branch {EXPERIMENT_BRANCH} so this run starts from current main")
except NotFoundException:
    pass

experiment = repo.branch(EXPERIMENT_BRANCH).create(source_reference="main")
print(f"Created branch {EXPERIMENT_BRANCH} from current main")

catalog_v2 = catalog_v1.copy()
haze_proxy = (catalog_v2["band_red"] > 0.55) & (catalog_v2["band_nir"] > 0.55)
catalog_v2["cloud_mask_v2"] = (
    (catalog_v2["cloud_mask_v1"] == 1) | haze_proxy
).astype("int8")

n_v1 = int(catalog_v2["cloud_mask_v1"].sum())
n_v2 = int(catalog_v2["cloud_mask_v2"].sum())
print(f"v1 flagged {n_v1} tiles, v2 flagged {n_v2} tiles (+{n_v2 - n_v1} new flags)")

experiment.object("tiles/catalog.parquet").upload(data=df_to_parquet_bytes(catalog_v2))
experiment_commit = experiment.commit(message="Add cloud_mask_v2 with haze proxy")
print(f"{EXPERIMENT_BRANCH} now at commit {experiment_commit.get_commit().id[:10]}")
print(f"main still at commit             {main.get_commit().id[:10]}")

## 4. Diff the branch against `main`

lakeFS exposes diffs at the object level. For Parquet, that tells you which files changed, not which rows; the cell below does both: a lakeFS-level diff and a row-level pandas comparison.

In [ ]:
print("=== Object-level diff (lakeFS) ===")
for entry in main.diff(other_ref=EXPERIMENT_BRANCH):
    print(f"  {entry.type:10s}  {entry.path}")

print("\n=== Row-level diff (pandas, for the catalog Parquet) ===")
main_bytes = main.object("tiles/catalog.parquet").reader().read()
exp_bytes = experiment.object("tiles/catalog.parquet").reader().read()
main_df = pq.read_table(io.BytesIO(main_bytes)).to_pandas()
exp_df = pq.read_table(io.BytesIO(exp_bytes)).to_pandas()
added_cols = set(exp_df.columns) - set(main_df.columns)
removed_cols = set(main_df.columns) - set(exp_df.columns)
print(f"  rows on main:       {len(main_df)}")
print(f"  rows on experiment: {len(exp_df)}")
print(f"  columns added:      {sorted(added_cols)}")
print(f"  columns removed:    {sorted(removed_cols)}")

## 5. Merge the experiment into `main`

The experiment looked good. Promote it to `main`. lakeFS merge is metadata-only; the underlying B2 objects do not move.

In [ ]:
merge_ref = experiment.merge_into("main", message="Promote cloud_mask_v2 to main")
post_merge_commit_id = main.get_commit().id
print(f"Merge commit: {merge_ref[:10] if isinstance(merge_ref, str) else merge_ref.id[:10]}")
print(f"main now at:  {post_merge_commit_id[:10]}")

main_after = pq.read_table(
    io.BytesIO(main.object("tiles/catalog.parquet").reader().read())
).to_pandas()
print(f"main catalog columns after merge: {list(main_after.columns)}")

## 6. Revert the merge

On second thought, v2 has too many false positives. Revert the merge commit on `main`. lakeFS does not rewrite history; instead it adds a new commit that re-applies the pre-merge state. Both the original merge and the revert stay visible in `main`'s history, and you can promote `experiment` again later by merging it once more.

In [ ]:
main.reset_changes()
main.revert(reference=post_merge_commit_id, parent_number=1)
print(f"main now at: {main.get_commit().id[:10]}")

main_after_revert = pq.read_table(
    io.BytesIO(main.object("tiles/catalog.parquet").reader().read())
).to_pandas()
print(f"main catalog columns after revert: {list(main_after_revert.columns)}")
print(f"v2 column gone: {'cloud_mask_v2' not in main_after_revert.columns}")

Both the merge and the revert are visible in `main`'s commit history. Run `lakectl log lakefs://${REPO_NAME}/main` (or `main.log()` from the SDK) to confirm.

## 7. (Optional) Inspect the B2 bucket directly

lakeFS lays out objects in B2 under a content-addressed scheme: `${STORAGE_NAMESPACE}/data/<hash-prefix>/<hash-suffix>`. The cell below uses `boto3` against B2 to show the actual key layout. It is purely educational; in normal use you should never read or write these objects directly. Skip this section if `STORAGE_NAMESPACE` is unset (i.e. neither `LAKEFS_NB_STORAGE_NAMESPACE` nor `B2_LAKEFS_DATA_BUCKET` was supplied).

Per the repo's `CLAUDE.md`, every boto3 client targeting B2 must set `user_agent_extra` and pin `region_name` explicitly.

In [ ]:
import boto3
from botocore.config import Config
from urllib.parse import urlparse

if not STORAGE_NAMESPACE:
    print("STORAGE_NAMESPACE is empty; skipping verification.")
else:
    parsed = urlparse(STORAGE_NAMESPACE)
    listing_bucket = parsed.netloc
    listing_prefix = parsed.path.lstrip("/") + "/"
    if B2_LAKEFS_DATA_BUCKET and B2_LAKEFS_DATA_BUCKET != listing_bucket:
        print(
            f"Note: STORAGE_NAMESPACE points at {listing_bucket!r}; "
            f"using that, not B2_LAKEFS_DATA_BUCKET={B2_LAKEFS_DATA_BUCKET!r}."
        )
    aws_access_key_id = os.environ.get("AWS_ACCESS_KEY_ID", "").strip() or input("AWS_ACCESS_KEY_ID (B2 keyID): ").strip()
    aws_secret_access_key = (
        os.environ.get("AWS_SECRET_ACCESS_KEY", "").strip()
        or getpass.getpass("AWS_SECRET_ACCESS_KEY (B2 applicationKey): ").strip()
    )
    b2 = boto3.client(
        "s3",
        endpoint_url=AWS_ENDPOINT_URL_S3,
        region_name=PRIVATE_B2_REGION,
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        config=Config(
            signature_version="s3v4",
            user_agent_extra="b2-notebook-lakefs",
            s3={"addressing_style": "path"},
        ),
    )
    resp = b2.list_objects_v2(
        Bucket=listing_bucket,
        Prefix=listing_prefix,
        MaxKeys=10,
    )
    print(f"First {len(resp.get('Contents', []))} keys under {listing_prefix!r} in {listing_bucket}:")
    for obj in resp.get("Contents", []):
        print(f"  {obj['Size']:>10}  {obj['Key']}")

## Cleanup

Two layers to clean up: the lakeFS repo and the B2 bucket contents lakeFS wrote underneath it. Substitute your own `REPO_NAME` and `STORAGE_NAMESPACE` values.

* **lakeFS repo.** `repo.delete()` from this SDK, or `lakectl repo delete lakefs://${REPO_NAME} --yes` from the CLI. This removes the lakeFS metadata. The B2 objects remain in place.
* **B2 objects.** The B2 console has a "Delete All Files" action under the bucket settings, or use `aws s3 rm --recursive ${STORAGE_NAMESPACE}/ --endpoint-url ${AWS_ENDPOINT_URL_S3}` to scope the delete to just this repo's prefix.

When you tear down the `lakefs-on-b2-quickstart` Compose stack with `docker compose down -v`, the Postgres metadata is wiped but the underlying B2 objects survive; clean them up explicitly if you do not want them.